# 🚀 Notebook do Professor (Demo) — Aula 02: Memória conversacional Buffer, Summary e TokenBuffer

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 02/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🧠 3 tipos de memória**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Tornar a chain da Aula 01 stateful — com memória persistida entre turnos — escolhendo o tipo certo para o domínio do grupo, e entender o trade-off de custo de tokens de cada abordagem.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — ConversationBufferMemory — o mais simples

In [6]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

OLLAMA_HOST: https://ollama.com
OLLAMA_API_KEY configurada: True


In [7]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

llm = ChatOllama(model="gpt-oss:120b")

# Memória: guarda TUDO — crescimento linear
memoria = ConversationBufferMemory(
    memory_key="history",        # chave no contexto do prompt
    return_messages=True,       # retorna como lista de mensagens
)

# ConversationChain: gerencia prompt + memória automaticamente
chat = ConversationChain(
    llm=llm,
    memory=memoria,
    verbose=True,              # mostra o prompt completo a cada turno
)

# Conversa — o histórico é mantido automaticamente
print(chat.predict(input="Meu nome é Ana."))
print(chat.predict(input="Qual é o meu nome?"))  # → "Seu nome é Ana"

# Inspecionar o que está na memória
print(memoria.load_memory_variables({}))
# → {"history": [HumanMessage(...), AIMessage(...), ...]}

/var/folders/ht/7xl05kvx4vxd90rtj1ym13yr0000gp/T/ipykernel_4340/1016511404.py:8: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(
/var/folders/ht/7xl05kvx4vxd90rtj1ym13yr0000gp/T/ipykernel_4340/1016511404.py:14: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  chat = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Meu nome é Ana.
AI:

> Finished chain.
Olá, Ana! 😊 Muito prazer em conhecê‑la. Eu sou o ChatGPT, um modelo de linguagem desenvolvido pela OpenAI, e estou aqui para conversar, responder dúvidas, ajudar com traduções, sugerir receitas, dar ideias de presentes – basicamente o que precisar.

Como está o seu dia até agora? Se quiser, podemos trocar ideias sobre algum assunto que lhe interesse: literatura brasileira, histórias de viagem, dicas de estudo, programação, cinema ou até curiosidades sobre a própria IA. Ah, e caso precise de algo em português, português de Portugal ou mesmo uma mistura divertida, é só dizer!

Se preferir, também podemos brincar de jogos d

### Slide 08 — ConversationSummaryMemory — resumo progressivo

In [8]:
from langchain_classic.memory import ConversationSummaryMemory

# Summary memory usa o próprio LLM para resumir
memoria_summary = ConversationSummaryMemory(
    llm=llm,                       # LLM usado para gerar o resumo
    memory_key="history",
    return_messages=True,
)

chat_summary = ConversationChain(
    llm=llm,
    memory=memoria_summary,
    verbose=True,
)

# Após vários turnos, inspecionar o resumo acumulado
chat_summary.predict(input="Meu nome é Ana e tenho 28 anos.")
chat_summary.predict(input="Trabalho como engenheira de dados.")
chat_summary.predict(input="Estou aprendendo LangChain.")

# O resumo é algo como:
# "A usuária Ana, 28 anos, engenheira de dados, está aprendendo LangChain."
# — independente de quantos turnos houve
print(memoria_summary.load_memory_variables({}))

/var/folders/ht/7xl05kvx4vxd90rtj1ym13yr0000gp/T/ipykernel_4340/1902099637.py:4: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria_summary = ConversationSummaryMemory(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[SystemMessage(content='', additional_kwargs={}, response_metadata={})]
Human: Meu nome é Ana e tenho 28 anos.
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[SystemMessage(content='The human introduces herself as Ana, a 28‑year‑old. The AI greets her, notes her age, and invites her to share her interests, hobbies, and goals. It then suggests several ideas—personalized music 

### Slide 09 — ConversationTokenBufferMemory — limite explícito de tokens

In [9]:
from langchain_classic.memory import ConversationTokenBufferMemory

# Limite de tokens para o histórico (excluindo o novo input)
memoria_token = ConversationTokenBufferMemory(
    llm=llm,                       # usado para contar tokens
    max_token_limit=500,           # máximo de tokens no histórico
    memory_key="history",
    return_messages=True,
)

chat_token = ConversationChain(
    llm=llm,
    memory=memoria_token,
    verbose=True,
)

# Com max_token_limit=500:
# - Turno 1–4: histórico completo (dentro do limite)
# - Turno 5: mensagem mais antiga é descartada automaticamente
# - Sempre mantém as mais recentes — janela deslizante

# Inspecionar quantos tokens estão no buffer agora
historico_atual = memoria_token.load_memory_variables({})
print(f"Mensagens no buffer: {len(historico_atual['history'])}")

Mensagens no buffer: 0


/var/folders/ht/7xl05kvx4vxd90rtj1ym13yr0000gp/T/ipykernel_4340/461406967.py:4: LangChainDeprecationWarning: The class `ConversationTokenBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria_token = ConversationTokenBufferMemory(


### Slide 10 — ConversationChain com system prompt customizado

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationTokenBufferMemory

# Template customizado — deve conter {history} e {input}
TEMPLATE = """Você é um assistente especialista em culinária brasileira.
Seja simpático e dê dicas práticas. Sempre que der uma receita,
liste os ingredientes de forma clara.

Histórico da conversa:
{history}

Usuário: {input}
Assistente:"""

prompt_custom = PromptTemplate(
    input_variables=["history", "input"],  # obrigatório para ConversationChain
    template=TEMPLATE,
)

chat_culinaria = ConversationChain(
    llm=llm,
    memory=ConversationTokenBufferMemory(llm=llm, max_token_limit=800),
    prompt=prompt_custom,
    verbose=False,
)

resp = chat_culinaria.predict(input="Como faço um feijão tropeiro?")
print(resp)

### Slide 14 — Inspecionar e manipular a memória em código

In [ ]:
# Ler o estado atual da memória (funciona em todos os tipos)
estado = memoria.load_memory_variables({})
print(estado)
# BufferMemory →    {"history": [HumanMessage, AIMessage, ...]}
# SummaryMemory →   {"history": "A usuária Ana é médica e..."}
# TokenBuffer →     {"history": [HumanMessage, AIMessage, ...]} (últimas N)

# Adicionar memória externamente (útil para testes)
memoria.save_context(
    {"input": "Qual é a capital do Brasil?"},
    {"output": "Brasília."},
)

# Limpar completamente a memória (ex: nova sessão de usuário)
memoria.clear()
print(memoria.load_memory_variables({}))  # → {"history": []}

# Acessar o buffer diretamente (Buffer e TokenBuffer)
if hasattr(memoria, "chat_memory"):
    print(f"Mensagens guardadas: {len(memoria.chat_memory.messages)}")

# Acessar o resumo (SummaryMemory)
if hasattr(memoria, "moving_summary_buffer"):
    print(f"Resumo atual: {memoria.moving_summary_buffer}")

### Slide 19 — Nota técnica — memória em LCEL moderno (LangChain 0.3+)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Armazena históricos por session_id (multi-usuário)
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# chain LCEL da Aula 01
chain_base = prompt | llm | StrOutputParser()

# Envolver com histórico — suporta múltiplas sessões simultâneas
chain_com_hist = RunnableWithMessageHistory(
    chain_base,
    get_session_history,
    input_messages_key="pergunta",
)

# Invocar com session_id — cada usuário tem seu histórico
chain_com_hist.invoke(
    {"pergunta": "Olá!"},
    config={"configurable": {"session_id": "usuario_123"}},
)

### Nota técnica — memória com `create_agent` (API atual do LangChain)

O LangChain reorganizou os agentes em torno de `create_agent`, que substitui `ConversationChain` como padrão recomendado. Memória de curto prazo (dentro de uma conversa) vem de um `checkpointer`; memória de longo prazo (entre conversas) vem de um `store`. `ConversationChain` continua funcionando — este bloco é só para referência, não é o padrão cobrado no CKP01.

In [ ]:
!pip install langchain langchain-ollama langgraph -q

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# create_agent + checkpointer é o padrão atual — substitui ConversationChain + memory
agent_moderno = create_agent(
    model=llm,                     # reaproveita o ChatOllama já criado acima
    tools=[],
    system_prompt="Você é um assistente prestativo.",
    checkpointer=InMemorySaver(),  # guarda o histórico por thread_id
)

# thread_id isola cada conversa — substitui o memory_key do ConversationChain
config = {"configurable": {"thread_id": "demo-aula-02"}}

resposta = agent_moderno.invoke(
    {"messages": [{"role": "user", "content": "Meu nome é Ana."}]},
    config,
)["messages"][-1].content
print(resposta)

resposta = agent_moderno.invoke(
    {"messages": [{"role": "user", "content": "Qual é o meu nome?"}]},
    config,
)["messages"][-1].content
print(resposta)  # → histórico mantido automaticamente pelo checkpointer

# Memória entre sessões diferentes (não só dentro de uma thread) usa Store:
# from langgraph.store.memory import InMemoryStore
# agent = create_agent(model=llm, tools=[...], store=InMemoryStore())
# docs.langchain.com/oss/python/langchain/long-term-memory

### Slide 22 — Python novo desta aula

In [ ]:
# 1. Importação de múltiplos símbolos com parênteses
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
)

# 2. Instância com parâmetros nomeados — keyword arguments
mem = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=800,
    return_messages=True,
)

# 3. hasattr() — verificar se objeto tem um atributo
if hasattr(mem, "chat_memory"):
    print(mem.chat_memory.messages)

# 4. Chamada .predict() — método de conveniência da ConversationChain
resp = chat.predict(input="Minha pergunta")
# equivale a chat.invoke({"input": "Minha pergunta"})["response"]

# 5. Dict com chave computed (f-string como key não é pythônico — use variável)
estado = mem.load_memory_variables({})  # {} = nenhum input extra
historico = estado["history"]

# 6. save_context() — injetar memória manualmente (útil em testes)
mem.save_context(
    {"input":  "pergunta do usuário"},
    {"output": "resposta do modelo"},
)

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 02 · 2º Semestre**  
### Chatbot com memória — domínio do grupo ★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 4 lacunas do notebook — template com persona do domínio, tipo de memória, montagem da ConversationChain e 5 turnos de teste.
2. Justifique a escolha do tipo de memória em comentário no notebook. Critérios: conversas do domínio tendem a ser curtas ou longas? O usuário vai referenciar informações do início da conversa?
3. Teste o limite da memória: após 5 turnos, pergunte algo que dependia do turno 1. A resposta ainda é coerente?
4. Desafio: adicione um segundo chat com outro tipo de memória e compare as respostas para a mesma conversa. Qual deu respostas mais coerentes no seu domínio?

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: f-string com persona + "Histórico:\n{history}\n\nUsuário: {input}\nAssistente:"
>
> Lacuna 2: ConversationTokenBufferMemory(llm=llm, max_token_limit=800) — boa para a maioria dos domínios
>
> Lacuna 3: llm=llm, memory=memoria, prompt=prompt
>
> Lacuna 4: 5 perguntas reais do domínio escolhido na Aula 01

> **💡 Dica de escolha:**
>
> domínios de suporte técnico ou jurídico → SummaryMemory (conversas longas, detalhes importam mas o resumo serve). Domínios de receitas ou jogos → TokenBuffer (conversas mais curtas, últimas mensagens são as mais relevantes).

In [ ]:
!pip install langchain langchain-ollama langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationTokenBufferMemory,
)
from langchain_classic.chains import ConversationChain
from langchain_core.prompts import PromptTemplate
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm = ChatOllama(model="gpt-oss:120b")

# 👉 LACUNA 1: escreva o template com a persona do domínio do grupo
# Lembre: deve conter {history} e {input}
TEMPLATE = ___

prompt = PromptTemplate(input_variables=["history", "input"], template=TEMPLATE)

# 👉 LACUNA 2: escolha o tipo de memória para o domínio do grupo
# Justifique em comentário: por que este tipo e não os outros?
memoria = ___  # Buffer, Summary ou TokenBuffer

# 👉 LACUNA 3: monte a ConversationChain com llm, memory e prompt
chat = ConversationChain(___)

# 👉 LACUNA 4: teste com 5 turnos do domínio do grupo
for pergunta in [___, ___, ___, ___, ___]:
    print(f"R: {chat.predict(input=pergunta)}\n")

# Inspecionar o estado final da memória
print(memoria.load_memory_variables({}))

## 📚 Referências da aula

- Docs LangChain — Memory: ConversationBufferMemory, SummaryMemory, TokenBufferMemory. python.langchain.com/docs/modules/memory
- Docs LangChain — ConversationChain: combinar memória com prompt customizado. python.langchain.com/docs/modules/chains
- Docs LangChain — RunnableWithMessageHistory: abordagem moderna para memória em LCEL (0.3+). python.langchain.com/docs/expression_language/how_to/message_history
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3: Memória em agentes — a relevância do histórico para tomada de decisão.
- Paper Brown, T. et al. — "Language Models are Few-Shot Learners." NeurIPS, 2020. Seção sobre context window e por que o tamanho do contexto impacta diretamente o custo e a qualidade. arxiv.org/abs/2005.14165
- Ebook Freed, A.; Jacobs, C.; Rózsa, E. — Effective Conversational AI. Manning, 2025. Cap. 9: Harnessing Context for an Adaptive Virtual Assistant Experience — a distinção entre session history e persistent user history que fundamenta esta aula.

---

**→ Próxima Aula — Aula 03 · 17/08** — Structured output e Pydantic v2
  
Forçar o LLM a responder com um schema garantido. CKP01 R3 entregue.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*